In [ ]:
from dotenv import load_dotenv
from typing import TypedDict, List, Annotated
from langgraph.graph import StateGraph, START, END, add_messages
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from  langgraph.prebuilt import ToolNode, tools_condition
from IPython.display import Image, display
from langgraph.checkpoint.memory import InMemorySaver  

load_dotenv()

In [ ]:
# initialize the LLM
llm = init_chat_model("llama-3.1-8b-instant", model_provider="groq")


In [ ]:
# mock external API

@tool
def get_stock_price(stock_symbol: str) -> float:
    """Returns the current stock price for the given stock symbol. In a real implementation, this would call an external API to get live stock prices.
    :param stock_symbol: The stock symbol to get the price for (e.g., "AAPL" for Apple Inc.)
    :return: The current stock price as a float
    """
        
    mock_prices = {
        "AAPL": 150.0,
        "GOOGL": 200.0,
        "AMZN": 300.0,
        "MSFT": 400.0,
    }
    return mock_prices.get(stock_symbol.upper(), 100.0)

tools= [get_stock_price]

# parallel_tool_calls=False -- forces one tool call at a time; works around small model parallel call issues
llm_with_tools = llm.bind_tools(tools, parallel_tool_calls=False)


In [ ]:
# graph state
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]

# graph node function
def chatbot(state: ChatState)-> ChatState:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

builder = StateGraph(ChatState)
builder.add_node("chatbot_node", chatbot)
builder.add_node("tools", ToolNode(tools))  # node must be named "tools" -- tools_condition routes to "tools" by default

builder.add_edge(START, "chatbot_node")
builder.add_conditional_edges("chatbot_node", tools_condition)  # routes to "tools" or END
builder.add_edge("tools", "chatbot_node")  
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
# Agentic - requires tool use
msg2 = {"role":"user", "content": "what is the sum of GOOGL stock and AAPL stock price?"}
respState = graph.invoke({"messages": [msg2]}) 
respState["messages"]

In [ ]:
# follow up invoke to test memory of past invoke and tool calls in the same conversation
msg3 = {"role":"user", "content": "Add AMZN stock price to previous sum of stocks?"}
respState = graph.invoke({"messages": [msg3]}) 
respState["messages"]

In [ ]:
# add In memory to have state persist across invokes in the same conversation
inMem = InMemorySaver()
graph_with_memory = builder.compile(checkpointer=inMem)

In [ ]:
memConfigStock = {'configurable':{'thread_id': 'stock_chat_thread'}}

# invoke with memory
msg3 = {"role":"user", "content": "what is the sum of GOOGL stock and AAPL stock price?"}
respState =graph_with_memory.invoke({"messages": [msg3]},config=memConfigStock) 
respState["messages"]

In [ ]:
# follow up invoke to test memory of past invoke and tool calls in the same conversation
msg4 = {"role":"user", "content": "Add AMZN stock price to previous sum of stocks?"}
respState = graph_with_memory.invoke({"messages": [msg4]},config=memConfigStock) 
respState["messages"]